# Assignment 2
## P+7 (Oulipian language modelling)

### Background

The Oulipo (*Ouvroir de Littérature Potentielle*, or “Workshop of Potential Literature”) is a French literary group founded in 1960 by writer Raymond Queneau and mathematician François Le Lionnais. The group focuses on using rules and constraints in writing as a way to spark creativity. Rather than seeing constraints as obstacles, Oulipians treat them as tools to inspire new forms of storytelling and poetry. Their work combines mathematics, language, and playfulness, making their approach both unique and influential in modern literature.

One of the most famous Oulipian writers is Georges Perec, who is known for his creative use of constraints. His novel [*La Disparition* (“A Void”) is written entirely without the letter "e,"](https://archive.org/details/void00pere/mode/2up) which is especially challenging given how common "e" is in French. Perec’s writing often plays with the structure of language in surprising ways.



![George Perec](https://upload.wikimedia.org/wikipedia/commons/7/76/Myart_georges-perec_1978.jpg)

(George Perec, 1978. From Wikidata)

<!-- <img src="https://media.vigliensoni.com/clips/CART498/perec-01.jpg" width="800"> -->


One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

## The N+7 technique

One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

The N+7 technique process is straightforward

- **Start with a text**. Choose any text—this could be a poem, a sentence, or a passage.

- **Use a dictionary**. Have a dictionary (or word list) handy.

- **Replace each noun**. For every substantive noun in the original text,
 replace it with the noun appearing seven nouns away in the dictionary. If the end of the dictionary is reached, you can loop back to the beginning.

- **Maintain grammar**. Ensure the new text maintains grammatical correctness as much as possible, though the results often turn out  surreal and nonsensical.

For example, using the N+7 technique with a standard English dictionary for the original sentence:

*One must have a mind of winter*.

- ”mind” → 7 nouns after “mind” is “minimum” or “mindset” (depending on the dictionary)
- “winter” → 7 nouns after  “wishing” or “wipe”

Then, after applying the technique the results are:

- Original: “One must have a mind of winter”
- Dictionary one: “One must have a minimum of wishing”
- Dictionary two: “One must have a mindset of wipe”

[Online example](http://www.spoonbill.org/n+7/)

### Assignment and deliverables

For this assignment, you will create a variation of the N+7 technique we will name P+7. Using the GPT-2 language model, you will replace the last word of each line from *The Snow Man* with the token that has the seventh-highest probability according to the model’s predictions.

By the end of this assignment, submit a link to a GitHub repository named `CART498-GenAI` containing a folder labelled `A02` with the following items:

- A version of the text processed with your P+7 technique, saved as a `.txt` file.
- A second version of the text processed with `P+x`. Choose an `x` value that produces the funniest, wittiest, or most absurd version of the original text. Save this as a .txt file, and include the `x` value in the filename (e.g., `P+23.txt` or `P+12.txt`).
- A Python notebook with the script used to generate your P+7 and P+x transformations using the GPT-2 language model.
- A short reflection (250–350 words) explaining how altering the `x` value impacted the output of your P+x version. Additionally, discuss how you would implement a P+7 technique in which all nouns are replaced with their seventh-highest probability alternatives.

### *The Snow Man*
by Wallace Stevens (1879-1955)


> One must have a mind of winter  
> To regard the frost and the boughs  
> Of the pine-trees crusted with snow;  
> And have been cold a long time  
> To behold the junipers shagged with ice,  
> The spruces rough in the distant glitter  
> Of the January sun; and not to think  
> Of any misery in the sound of the wind,  
> In the sound of a few leaves,  
> Which is the sound of the land  
> Full of the same wind  
> That is blowing in the same bare place  
> For the listener, who listens in the snow,  
> And, nothing himself, beholds  
> Nothing that is not there and the nothing that is.  




In [1]:
# global var
pPlusSeven = ""

In [54]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# global var
poem = [
    "One must have a mind of winter",
    "To regard the frost and the boughs",
    "Of the pine-trees crusted with snow;",
    "And have been cold a long time",
    "To behold the junipers shagged with ice,",
    "The spruces rough in the distant glitter",
    "Of the January sun; and not to think",
    "Of any misery in the sound of the wind,",
    "In the sound of a few leaves,",
    "Which is the sound of the land",
    "Full of the same wind",
    "That is blowing in the same bare place",
    "For the listener, who listens in the snow,",
    "And, nothing himself, beholds",
    "Nothing that is not there and the nothing that is."
]




# reuse earlier code
# (Generated in page for in class activity2, then edited)
def get_top_tokens_and_probabilities(model_name, prompt, num_tokens=100):
    """
    Loads a model and tokenizer, then returns the top N most probable next tokens
    and their probabilities for a given prompt.
    """
    # Using a cached tokenizer and model to avoid reloading in every iteration of the loop
    if 'tokenizer' not in get_top_tokens_and_probabilities.__dict__ or \
       get_top_tokens_and_probabilities.tokenizer.name_or_path != model_name:
        get_top_tokens_and_probabilities.tokenizer = AutoTokenizer.from_pretrained(model_name)
        get_top_tokens_and_probabilities.model = AutoModelForCausalLM.from_pretrained(model_name)
        get_top_tokens_and_probabilities.model.eval()

    tokenizer = get_top_tokens_and_probabilities.tokenizer
    model = get_top_tokens_and_probabilities.model

    inputs = tokenizer(prompt, return_tensors="pt")
    # print(inputs)
    # print(prompt)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits


    next_token_logits = logits[0, -1, :]
    probabilities = torch.softmax(next_token_logits, dim=-1)

    # Ensure num_tokens does not exceed |tokens|
    num_tokens = min(num_tokens, probabilities.shape[0])

    top_probs, top_indices = torch.topk(probabilities, num_tokens)

    results = []
    for prob, idx in zip(top_probs, top_indices):
        token = tokenizer.decode(idx)
        # print(token) # Removed for cleaner output
        results.append({"token": token, "probability": prob.item()})
    return results


new_poem_lines = [] # Output list


def poem_p_plus_n (poem, wordN, model_name):

  for line in poem:
      parses = line.split(" ")
      temp = parses[-1] # Original last word
      toadd = ""
      cond_full_word = False

      # Determine punctuation
      if temp.endswith(","):
          toadd = ","
      elif temp.endswith(";"):
          toadd = ";"
      elif temp.endswith("."):
          toadd = "."

      # remove last word (Section 1 )
      parses.pop()
      prompt_for_model = " ".join(parses)

      # Get the wordN'th most probable token
      initial_predictions = get_top_tokens_and_probabilities(model_name, prompt_for_model, num_tokens=wordN)
      generated_final_word = ""

      if initial_predictions and len(initial_predictions) >= wordN:
          # Start with the Nth token from the initial predictions
          current_word_segment = initial_predictions[wordN-1]['token']

          # Loop to complete the word if it's a sub-word
          # also allows non-words/run on words? to generate witch is funny
          while not cond_full_word:
              prompt_to_predict_next = prompt_for_model + " " + current_word_segment
              next_token_result = get_top_tokens_and_probabilities(model_name, prompt_to_predict_next, num_tokens=1)
              #
              if next_token_result and len(next_token_result) > 0:
                  predicted_next_token = next_token_result[0]['token']

                  # end cond_ for new words
                  if predicted_next_token.startswith(" ") or predicted_next_token.startswith("\n") or predicted_next_token.startswith(";")or predicted_next_token.startswith(",") or predicted_next_token.startswith("."):
                      # word isnt s
                      cond_full_word = True

                  else:
                      # not full -> itterate
                      current_word_segment += predicted_next_token
              else:
                  # base case for inf loop
                  cond_full_word = True

          generated_final_word = current_word_segment.strip()

      # Concatenate the new line
      transformed_line = " ".join(parses)
      transformed_line += " " + generated_final_word + toadd
      new_poem_lines.append(transformed_line)

  # The user wants to see the result, so print the entire transformed poem.
  pPlusSeven = "\n".join(new_poem_lines)
  out = ("Poem n+" + str(wordN) + ":"+"\n" + pPlusSeven)
  return out

# Define vars
model_name = "openai-community/gpt2"
wordN = 50000



pPlusSeven = poem_p_plus_n(poem, wordN, model_name)
print(pPlusSeven)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Poem n+50000:
One must have a mind of �"
To regard the frost and the elaide
Of the pine-trees crusted with intendent;
And have been cold a long eguish
To behold the junipers shagged with quez,
The spruces rough in the distant phabet
Of the January sun; and not to ヘライン
Of any misery in the sound of the anoia,
In the sound of a few onnaissance,
Which is the sound of the ーテル
Full of the same FORMATION
That is blowing in the same bare cifico
For the listener, who listens in the aiman,
And, nothing himself, ruary
Nothing that is not there and the nothing that UGC.


In [27]:
from google.colab import drive
drive.mount('/content/drive')
import os

output_dir = '/content/drive/MyDrive/CART498/A2'
os.makedirs(output_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
# file counter
counter = 0
print(pPlusSeven)

Poem n+49999:
One must have a mind of QL
To regard the frost and the LOGIC
Of the pine-trees crusted with izontal;
And have been cold a long IMAGES
To behold the junipers shagged with DIT,
The spruces rough in the distant アルテンド
Of the January sun; and not to arnaevate
Of any misery in the sound of the rafian's,
In the sound of a few assadoras,
Which is the sound of the advertising?
Full of the same rehensive
That is blowing in the same bare emisculated
For the listener, who listens in the ertodd,
And, nothing himself, veyard
Nothing that is not there and the nothing that uyomi.


In [53]:


# save prompt (and relevant context) to drive
prompt_filename = os.path.join(output_dir, f"N+{wordN}.txt")
with open(prompt_filename, "w") as f:
    f.write(pPlusSeven)
    # f.write("Poem n+" + str(wordN) + ":\n" + pPlusSeven)

print(f"Text saved to: {prompt_filename}")

Text saved to: /content/drive/MyDrive/CART498/A2/N+49999.txt
